In [5]:
import inspect
import typing

from langchain_core.language_models import BaseChatModel
from langchain_core.language_models.base import LanguageModelInput
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


# The exercise calls `movie_chain.invoke({"movie_name": "The Matrix"})`. 


## Step 1.  So the first thing in the chain must accept a dict as input.

## Step 2. Does model (a ChatOpenAI) accept a raw dict as input? Check:

In [6]:
(inspect.signature(BaseChatModel.invoke).parameters, 
inspect.signature(BaseChatModel.invoke)._return_annotation)
# nao aceita dict

(mappingproxy({'self': <Parameter "self">,
               'input': <Parameter "input: 'LanguageModelInput'">,
               'config': <Parameter "config: 'Optional[RunnableConfig]' = None">,
               'stop': <Parameter "stop: 'Optional[list[str]]' = None">,
               'kwargs': <Parameter "**kwargs: 'Any'">}),
 'BaseMessage')

In [7]:
(inspect.signature(ChatOpenAI.invoke).parameters, 
inspect.signature(ChatOpenAI.invoke)._return_annotation)
# nao aceita dict

(mappingproxy({'self': <Parameter "self">,
               'input': <Parameter "input: 'LanguageModelInput'">,
               'config': <Parameter "config: 'Optional[RunnableConfig]' = None">,
               'stop': <Parameter "stop: 'Optional[list[str]]' = None">,
               'kwargs': <Parameter "**kwargs: 'Any'">}),
 'BaseMessage')

## Step 3. What is LanguageModelInput?

In [8]:
print(LanguageModelInput)
# No dict in that union. So model can't be first in the chain — it doesn't accept what you're calling invoke with.

typing.Union[langchain_core.prompt_values.PromptValue, str, collections.abc.Sequence[typing.Union[langchain_core.messages.base.BaseMessage, list[str], tuple[str, str], str, dict[str, typing.Any]]]]


## Step 4. Something has to go before model that accepts a dict and outputs one of PromptValue/str/message-sequence. 
> Search langchain_core.prompts for a class whose job description matches "fill a dict of variables into a template."

## Step 5. Check PromptTemplate.invoke's types:

In [9]:
typing.get_type_hints(PromptTemplate.invoke)
# input: dict  ->  return: PromptValue

{'input': dict,
 'config': typing.Optional[langchain_core.runnables.config.RunnableConfig],
 'kwargs': typing.Any,
 'return': langchain_core.prompt_values.PromptValue}

> Conclusion
>> as PromptTemplate accepts a dict, it can be first in the chain. But BaseChatModel does not accept a dict, so it cannot be first in the chain.
>>> As PromptTemplate outputs PromptValue and model accepts PromptValue as input, model can be chained after: prompt_template | model.

In [10]:
typing.get_type_hints(JsonOutputParser)

{'name': typing.Optional[str],
 'diff': bool,
 'pydantic_object': typing.Optional[type[~TBaseModel]]}

In [11]:
( inspect.signature(JsonOutputParser.invoke).parameters,
inspect.signature(JsonOutputParser.invoke)._return_annotation)

(mappingproxy({'self': <Parameter "self">,
               'input': <Parameter "input: 'Union[str, BaseMessage]'">,
               'config': <Parameter "config: 'Optional[RunnableConfig]' = None">,
               'kwargs': <Parameter "**kwargs: 'Any'">}),
 'T')

> Conclusion
>> As model outputs a BaseMessage, and the parser takes a BaseMessage as input, they can be chained.
>>> Prompt_Template | model | JsonOutputParser